#**LAB 8 (21 Mar- 22 Mar 2025)**
#TOPICS COVERED : Recurrent Neural Networks


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np


[https://karpathy.github.io/2015/05/21/rnn-effectiveness/](https://karpathy.github.io/2015/05/21/rnn-effectiveness/)

The notebook provides a comprehensive overview of building and training a basic RNN for a character-level language modeling task. It covers data preprocessing, model definition, training, and using the model for predictions.

In [2]:
text=['i like cake','she likes Pizza','she loves flower']

In [3]:
chars = set(''.join(text))

In [4]:
chars

{' ',
 'P',
 'a',
 'c',
 'e',
 'f',
 'h',
 'i',
 'k',
 'l',
 'o',
 'r',
 's',
 'v',
 'w',
 'z'}

In [5]:
int_map = dict(enumerate(chars))

In [6]:
int_map

{0: 'k',
 1: 'v',
 2: 'r',
 3: ' ',
 4: 'P',
 5: 'a',
 6: 'h',
 7: 'l',
 8: 'f',
 9: 'o',
 10: 'w',
 11: 'e',
 12: 'c',
 13: 'z',
 14: 'i',
 15: 's'}

In [7]:
char_map = {char:ind for ind, char in int_map.items()}

In [8]:
char_map

{'k': 0,
 'v': 1,
 'r': 2,
 ' ': 3,
 'P': 4,
 'a': 5,
 'h': 6,
 'l': 7,
 'f': 8,
 'o': 9,
 'w': 10,
 'e': 11,
 'c': 12,
 'z': 13,
 'i': 14,
 's': 15}

In [9]:
num_unique_chars = len(char_map)

In [10]:
num_unique_chars

16

In [11]:
maxlen = len(max(text, key=len))

In [12]:
maxlen

16

# the items must be of the same dims if we want to stack them into a batch

for example, say we have a dataset of images

and say our batch size is 30

256x256

512x512

(30, 3, 256, 256)

In [13]:
# iterating over my sentences in the dataset
for i in range(len(text)):
  while(len(text[i]))<maxlen:
    text[i] += ' '

In [14]:
text

['i like cake     ', 'she likes Pizza ', 'she loves flower']

In [15]:
input_seq = list()
target_seq = list()


for i in range(len(text)):
  input_seq.append(text[i][:-1])
  target_seq.append(text[i][1:])


In [16]:
input_seq

['i like cake    ', 'she likes Pizza', 'she loves flowe']

In [17]:
target_seq

[' like cake     ', 'he likes Pizza ', 'he loves flower']

In [18]:
for i in range(len(text)):
  input_seq[i] = [char_map[character] for character in input_seq[i]]
  target_seq[i] = [char_map[character] for character in target_seq[i]]


In [20]:
input_seq


[[14, 3, 7, 14, 0, 11, 3, 12, 5, 0, 11, 3, 3, 3, 3],
 [15, 6, 11, 3, 7, 14, 0, 11, 15, 3, 4, 14, 13, 13, 5],
 [15, 6, 11, 3, 7, 9, 1, 11, 15, 3, 8, 7, 9, 10, 11]]

In [19]:
target_seq


[[3, 7, 14, 0, 11, 3, 12, 5, 0, 11, 3, 3, 3, 3, 3],
 [6, 11, 3, 7, 14, 0, 11, 15, 3, 4, 14, 13, 13, 5, 3],
 [6, 11, 3, 7, 9, 1, 11, 15, 3, 8, 7, 9, 10, 11, 2]]

In [21]:
# you want to get the one-hot embedding/vector corresponding to 4

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [22]:
import numpy as np

import torch
from torch import nn


num_of_sentences x num_characters_in_sentence x length_of_one_hot_vector

In [23]:
def one_hot_encode(sequence, num_unique_chars, seq_len, batch_size):

  features = np.zeros((batch_size, seq_len, num_unique_chars), dtype=np.float32)
  # for each sentence
  for i in range(batch_size):
    # for each character in a sentence
    for u in range(seq_len):
      features[i, u, sequence[i][u]] = 1

  return features


In [24]:
batch_size = len(text)
seq_len = maxlen - 1

In [25]:
input_seq = one_hot_encode(input_seq, num_unique_chars, seq_len, batch_size)

In [26]:
type(input_seq)

numpy.ndarray

In [27]:
input_seq = torch.from_numpy(input_seq)
target_seq = torch.Tensor(target_seq)

In [28]:
device = torch.device('cuda')

# Making the model

In [29]:
class Model(nn.Module):
  def __init__(self, input_size, output_size, hidden_dim, n_layers):
    super().__init__()
    self.hidden_dim = hidden_dim
    self.n_layers = n_layers

    self.rnn = nn.RNN(input_size, hidden_dim, n_layers, batch_first = True)
    self.fc = nn.Linear(hidden_dim, output_size)

  def init_hidden(self, batch_size):
    hidden = torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)
    return hidden

  def forward(self, x):
    batch_size = x.shape[0]
    hidden = self.init_hidden(batch_size)

    out, hidden  = self.rnn(x, hidden)

    out = self.fc(out)

    return out, hidden


In [30]:
model = Model(input_size = num_unique_chars, output_size = num_unique_chars, hidden_dim = 12, n_layers = 1)

In [31]:
model = model.to(device)

In [32]:
n_epochs = 100
lr = 0.01

In [33]:
loss = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [34]:
input_seq = input_seq.to(device)

# Training the model

In [35]:
for epoch in range(1, n_epochs + 1):
  optimizer.zero_grad()

  output, hidden = model(input_seq)

  output = output.to(device)
  target_seq = target_seq.to(device)

  epoch_loss = loss(output.view(-1, output.shape[-1]), target_seq.view(-1).long())

  epoch_loss.backward()
  optimizer.step()

  if epoch % 10 == 0:
    print("Epoch: {}/{}............".format(epoch, n_epochs), end = ' ')
    print("Loss: {:.4f}".format(epoch_loss.item()))

Epoch: 10/100............ Loss: 2.4286
Epoch: 20/100............ Loss: 2.1306
Epoch: 30/100............ Loss: 1.7283
Epoch: 40/100............ Loss: 1.3376
Epoch: 50/100............ Loss: 1.0337
Epoch: 60/100............ Loss: 0.7868
Epoch: 70/100............ Loss: 0.5896
Epoch: 80/100............ Loss: 0.4463
Epoch: 90/100............ Loss: 0.3420
Epoch: 100/100............ Loss: 0.2646


In [36]:
output, hidden = model(input_seq)

In [37]:
output.shape

torch.Size([3, 15, 16])

In [38]:
target_seq.shape

torch.Size([3, 15])

In [39]:
target_seq.view(-1)

tensor([ 3.,  7., 14.,  0., 11.,  3., 12.,  5.,  0., 11.,  3.,  3.,  3.,  3.,
         3.,  6., 11.,  3.,  7., 14.,  0., 11., 15.,  3.,  4., 14., 13., 13.,
         5.,  3.,  6., 11.,  3.,  7.,  9.,  1., 11., 15.,  3.,  8.,  7.,  9.,
        10., 11.,  2.], device='cuda:0')

# Get predictions from our trained model

In [40]:
# characters = ['h', 'e', 'y']
def predict(model, characters):
  characters = np.array([[char_map[c] for c in characters]])
  characters = one_hot_encode(characters, num_unique_chars, characters.shape[1], 1)
  characters = torch.from_numpy(characters)
  characters = characters.to(device)

  model.eval()

  out, hidden = model(characters)

  prob = nn.functional.softmax(torch.squeeze(out, dim=0)[-1], dim=0)

  char_ind = torch.argmax(prob, dim=0)

  return int_map[char_ind.item()], hidden

In [41]:
def sample(model, out_len, start):

  model.eval()

  start = start.lower()

  chars = [ch for ch in start]

  size = out_len - len(chars)

  for _ in range(size):
    char, h = predict(model, chars)
    chars.append(char)

  return ''.join(chars)

In [128]:
sample(model, 6, 'she')

'she likes'

In [ ]:
# [0.01, 0.02, 0.3, 0.04, .....]
# you apply softmax to this -> you get a list of probabilites
# you use torch.argmax to get the index corresponding to the max value
# say our max value -> 3
# we use the int_map to convert 3 to 'i'


In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [45]:
# Sample text data
text=['i like cake','she likes Pizza','she loves flower']

In [46]:
# Tokenize at word level
words = set(" ".join(text).split())
word2idx = {word: idx for idx, word in enumerate(words)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)

In [47]:
# Convert sentences to sequences
sequences = [[word2idx[word] for word in sentence.split()] for sentence in text]

In [48]:
# Create input-output pairs
inputs = []
targets = []
for seq in sequences:
    for i in range(len(seq) - 1):
        inputs.append(seq[:i+1])
        targets.append(seq[i+1])

In [49]:
# Padding sequences to have the same length
max_len = max(len(seq) for seq in inputs)
inputs = [seq + [0] * (max_len - len(seq)) for seq in inputs]

tensor_inputs = torch.tensor(inputs, dtype=torch.long)
tensor_targets = torch.tensor(targets, dtype=torch.long)


In [50]:
# Dataset class
class WordDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

In [51]:
# DataLoader
dataset = WordDataset(tensor_inputs, tensor_targets)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [52]:
# Define Word-Level RNN model
class WordLevelRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(WordLevelRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.fc(hidden.squeeze(0))
        return out

In [75]:
# Model, Loss, Optimizer
embedding_dim = 10
hidden_dim = 20
model = WordLevelRNN(vocab_size, embedding_dim, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [76]:
# Training loop
epochs = 100
for epoch in range(epochs):
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 2.2253589630126953
Epoch 10, Loss: 0.8342220187187195
Epoch 20, Loss: 0.038546279072761536
Epoch 30, Loss: 0.01646791398525238
Epoch 40, Loss: 0.7073342800140381
Epoch 50, Loss: 0.3900395333766937
Epoch 60, Loss: 0.006949242204427719
Epoch 70, Loss: 0.3762397766113281
Epoch 80, Loss: 0.3804260790348053
Epoch 90, Loss: 0.0038142185658216476


In [77]:

# Prediction function
def predict_next_word(model, sentence, word2idx, idx2word):
    words = sentence.split()
    seq = [word2idx[word] for word in words if word in word2idx]
    seq = torch.tensor([seq + [0] * (max_len - len(seq))], dtype=torch.long)
    output = model(seq)
    predicted_idx = torch.argmax(output, dim=1).item()
    return idx2word[predicted_idx]

In [78]:
# Example usage
print(predict_next_word(model, "i like", word2idx, idx2word))

cake


In [79]:
# Example usage
print(predict_next_word(model, "she", word2idx, idx2word))

likes


In [80]:
# Example usage
print(predict_next_word(model, "she loves ", word2idx, idx2word))

flower


In [81]:
# Example usage
print(predict_next_word(model, "i like cake she likes", word2idx, idx2word))

Pizza


Train the model with the learning rate 0.001 ePOCH = 500 OPTIMIZER = sgd ONE MORE rnn LAYER

In [105]:
class WordLevelRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(WordLevelRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.fc(hidden[-1])
        return out

In [106]:
# Model, Loss, Optimizer
embedding_dim = 10
hidden_dim = 20
model = WordLevelRNN(vocab_size, embedding_dim, hidden_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [107]:
# Training loop
epochs = 500
for epoch in range(epochs):
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 2.2162117958068848
Epoch 10, Loss: 1.8474228382110596
Epoch 20, Loss: 1.5774965286254883
Epoch 30, Loss: 1.6825380325317383
Epoch 40, Loss: 1.2233083248138428
Epoch 50, Loss: 1.0260279178619385
Epoch 60, Loss: 0.6036220788955688
Epoch 70, Loss: 0.6766971349716187
Epoch 80, Loss: 0.6980779767036438
Epoch 90, Loss: 0.5411459803581238
Epoch 100, Loss: 0.617422878742218
Epoch 110, Loss: 0.5633144974708557
Epoch 120, Loss: 0.520258367061615
Epoch 130, Loss: 0.1591649204492569
Epoch 140, Loss: 0.453940212726593
Epoch 150, Loss: 0.505501925945282
Epoch 160, Loss: 0.45810025930404663
Epoch 170, Loss: 0.4244334101676941
Epoch 180, Loss: 0.41658344864845276
Epoch 190, Loss: 0.07326042652130127
Epoch 200, Loss: 0.06346144527196884
Epoch 210, Loss: 0.047370605170726776
Epoch 220, Loss: 0.4123559892177582
Epoch 230, Loss: 0.40682145953178406
Epoch 240, Loss: 0.04537493735551834
Epoch 250, Loss: 0.042110517621040344
Epoch 260, Loss: 0.03934800624847412
Epoch 270, Loss: 0.0373070873320

In [108]:

# Prediction function
def predict_next_word(model, sentence, word2idx, idx2word):
    words = sentence.split()
    seq = [word2idx[word] for word in words if word in word2idx]
    seq = torch.tensor([seq + [0] * (max_len - len(seq))], dtype=torch.long)
    output = model(seq)
    predicted_idx = torch.argmax(output, dim=1).item()
    return idx2word[predicted_idx]

In [109]:
print(predict_next_word(model, "i like", word2idx, idx2word))

cake


In [130]:
print(predict_next_word(model, "i", word2idx, idx2word))

like


In [129]:
print(predict_next_word(model, "i likes", word2idx, idx2word))

cake


In [111]:
print(predict_next_word(model, "she loves ", word2idx, idx2word))

flower


In [112]:
print(predict_next_word(model, "i like cake she likes", word2idx, idx2word))

Pizza
